# Plateau’s Problem for Enneper surface

This notebook provides an implementation of the Plateau's problem, which finds a minimal surface shape that connects a set of interfaces.
<!-- More details on this example, can be found in [our paper](https://arxiv.org/abs/2402.14009), Sections 4.1 and A.2. -->

### Imports and setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import GeneralNet
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.error_metrics import chamfer_div, compute_distance 
from util.visualization.utils_mesh import get_mesh
from training.modular.residuals import ResidualLibrary, ResidualTerm, compute_loss
from training.modular.optimizers import GaussNewton

torch.manual_seed(0)


device = 'cpu' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### True Surface

In [2]:
# Parametric equations for Enneper's minimal surface in polar coordinates
def enneper_surface_polar(r, phi):
    x = r * torch.cos(phi) - (1/3) * r**3 * torch.cos(3 * phi)
    y = r * torch.sin(phi) + (1/3) * r**3 * torch.sin(3 * phi)
    z = r**2 * torch.cos(2 * phi)
    return x, y, z
    
def enneper_level_set(v):
    x = v[:, 0]
    y = v[:, 1]
    z = v[:, 2]
    
    term1 = y**2 - x**2 + (4/3)*z + (4/9)*z**3
    term2 = y**2 - x**2 + (8/9)*z - z*(x**2 + y**2 + (8/9)*z**2)
    output = term1**3 - 3*z*term2**2
    return output

def sample_true_surface(n_samples):
    # Generate radial and angular coordinates
    r = torch.linspace(-r_max, r_max, n_samples, dtype=torch.float64)
    phi = torch.linspace(-torch.pi, torch.pi, 2*n_samples, dtype=torch.float64)
    
    # Create a grid of r and phi
    r, phi = torch.meshgrid(r, phi, indexing='ij')

    # Compute the x, y, z coordinates using the parametric equations
    x, y, z = enneper_surface_polar(r.flatten(), phi.flatten())
    points_on_surface = torch.vstack([x, y, z]).T

    return points_on_surface

# Bounds and number of samples
n = 1000
r_max = 0.9
phi = torch.linspace(-torch.pi, torch.pi, n, dtype=torch.float64)

# Generate boundary points with r constant and phi ranging from -pi to pi
r_constant = torch.full_like(phi, r_max, dtype=torch.float64)
x, y, z = enneper_surface_polar(r_constant, phi)
pts_boundary = torch.vstack([x, y, z]).T
pts_surface_true = sample_true_surface(64)

# Generate the mesh using the level set function
verts, faces = get_mesh(
    enneper_level_set, N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -2], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 2], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()


/opt/homebrew/Caskroom/miniforge/base/envs/ginns/lib/python3.12/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

### Pretraining

In [3]:
# Generate random training points
model = GeneralNet(ks=[3, 32, 32, 1])
model = model.double()
# Generate random training points
num_pretrain_samples = 10000
bounds = torch.tensor([[-1,1], [-1,1], [-1,1]], dtype=torch.float64)
pts_pretrain = torch.rand(num_pretrain_samples, 3, dtype=torch.float64) * (bounds[:,1] - bounds[:,0]) + bounds[:,0]


# Define pretraining loss
def pretrain_loss(model, params, pts):
    inputs = pts.to(dtype=torch.float64)
    targets = pts[:, 2].to(dtype=torch.float64)
    preds = model(inputs).squeeze(1)
    return 0.5 * (preds - targets).square().mean()

# Pretraining loop
pretrain_optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_pretrain_iters = 1000

for i in range(num_pretrain_iters):
    pretrain_optimizer.zero_grad()
    loss = pretrain_loss(model, model.params, pts_pretrain)
    loss.backward()
    pretrain_optimizer.step()
    
    if i % 100 == 0:
        print(f"Pretrain Iter {i}: Loss = {loss.item():.6f}")

print("Pretraining completed!")

verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-1.5, -1.5, -1], dtype=torch.float64),
    bbox_max=torch.tensor([1.5, 1.5, 1], dtype=torch.float64),
    chunks=2
)


fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)
fig.display()


Pretrain Iter 0: Loss = 0.141297
Pretrain Iter 100: Loss = 0.000277
Pretrain Iter 200: Loss = 0.000206
Pretrain Iter 300: Loss = 0.000168
Pretrain Iter 400: Loss = 0.000142
Pretrain Iter 500: Loss = 0.000123
Pretrain Iter 600: Loss = 0.000107
Pretrain Iter 700: Loss = 0.000094
Pretrain Iter 800: Loss = 0.000082
Pretrain Iter 900: Loss = 0.000071
Pretraining completed!


/opt/homebrew/Caskroom/miniforge/base/envs/ginns/lib/python3.12/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/ginns/lib/python3.12/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

### Main training loop

In [ ]:
res_lib = ResidualLibrary(model)

pts_eikonal = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
pts_eikonal = torch.cat((pts_eikonal, pts_boundary))

## Select the residual terms defining the specific problem
## (incl. weight, points, and (optional) target values)
res_terms = {
    "data":
        ResidualTerm(res_lib._data,           1.0,  pts_boundary, vals=0),
    "eikonal":
        ResidualTerm(res_lib._eikonal,        1e-3, pts_eikonal,        ),
    "mean_curvature":
        ResidualTerm(res_lib._mean_curvature, 1.0,  pts_boundary,       ),
}

In [ ]:
model = model.double()
params = model.params
loss_over_time = {}
distance_over_time = {}
chamfer_over_time = {}
best_loss = float('inf')

optim = GaussNewton(model, res_terms, lr=1e-1, regularization=1e-6, do_line_search=False)
# optim = torch.optim.Adam(model.parameters(), lr=1e-3)
start_time = time.time()
current_time = 0

for i in (pbar:=trange(100000)):
    # if current_time > 1200:
    if current_time > 600: # Shorter run for testing
        break
    optim.zero_grad()


    ## Update weights
    if i == 500:
        loss_weights = {"data": 1.0, "eikonal": 0.0, "mean_curvature": 1.0}
        for key in res_terms:
            res_terms[key].weight = loss_weights[key]

    ## Update surface points
    bound_limit = 1.0
    bounds = torch.tensor([[-bound_limit,bound_limit], [-bound_limit,bound_limit], [-bound_limit,bound_limit]], dtype=torch.float64)
    pts_surface = sample_model_surface_binsearch(model, pts_boundary, bounds)
    res_terms["mean_curvature"].points = pts_surface
    
    ## Evaluate the losses
    loss, unweighted_losses = compute_loss(params, res_terms, return_unweighted_losses=True)
        
    loss.backward()
    
    with torch.no_grad():
        loss_metric = unweighted_losses["data"] + unweighted_losses["mean_curvature"]

        current_time = time.time() - start_time
        loss_over_time[current_time] = loss_metric.item()
        distance_over_time[current_time] = compute_distance(model.double(), res_lib, enneper_level_set, pts_surface, pts_surface_true, 1.0)
        chamfer_over_time[current_time] = chamfer_div(model, res_lib, pts_surface_true)

        if loss_metric.item() < best_loss:
            best_loss = loss_metric.item()
            best_model_state = copy.deepcopy(model.state_dict())

        unweighted_losses_str = " ".join(f"{key}: {l.item():.2e}" for key, l in unweighted_losses.items())
        pbar.set_description(unweighted_losses_str + " "
                            f"error: {chamfer_over_time[current_time]:.2e} "
                            f"nof_pts: {len(pts_surface)}"
                            )
    optim.step()

# Optional: Load the best model after training
# if best_model_state is not None:
#     model.load_state_dict(best_model_state)
#     print(f"Best model loaded with loss {best_loss}")

plt.plot(loss_over_time.keys(), loss_over_time.values())
plt.semilogy()
plt.show()

  0%|          | 0/100000 [00:00<?, ?it/s]

/opt/homebrew/Caskroom/miniforge/base/envs/ginns/lib/python3.12/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1741947704867/work/aten/src/ATen/native/TensorShape.cpp:3638.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


KeyboardInterrupt: 

In [5]:
model = model.double()
params = model.params
loss_over_time = {}
distance_over_time = {}
chamfer_over_time = {}

optim = torch.optim.LBFGS(model.parameters(), lr=1e-1, max_iter=20, history_size=10)
start_time = time.time()
current_time = 0
best_loss = float('inf')
best_model_state = None

pbar = trange(100000)
for i in pbar:
    if current_time > 600:
        break

    # Update weights
    if i == 500:
        loss_weights = {"data": 1.0, "eikonal": 0.0, "mean_curvature": 1.0}
        for key in res_terms:
            res_terms[key].weight = loss_weights[key]

    # Update surface points
    bound_limit = 1.0
    bounds = torch.tensor([[-bound_limit, bound_limit], [-bound_limit, bound_limit], [-bound_limit, bound_limit]], dtype=torch.float64)
    pts_surface = sample_model_surface_binsearch(model, pts_boundary, bounds)
    res_terms["mean_curvature"].points = pts_surface

    def closure():
        optim.zero_grad()
        loss, _ = compute_loss(params, res_terms, return_unweighted_losses=True)
        loss.backward()
        return loss

    loss, unweighted_losses = compute_loss(params, res_terms, return_unweighted_losses=True)
    optim.step(closure)

    with torch.no_grad():
        loss_metric = unweighted_losses["data"] + unweighted_losses["mean_curvature"]
        current_time = time.time() - start_time
        loss_over_time[current_time] = loss_metric.item()
        distance_over_time[current_time] = compute_distance(model.double(), res_lib, enneper_level_set, pts_surface, pts_surface_true, 1.0)
        chamfer_over_time[current_time] = chamfer_div(model, res_lib, pts_surface_true)

        if loss_metric.item() < best_loss:
            best_loss = loss_metric.item()
            best_model_state = copy.deepcopy(model.state_dict())

        unweighted_losses_str = " ".join(f"{key}: {l.item():.2e}" for key, l in unweighted_losses.items())
        pbar.set_description(unweighted_losses_str + " "
                            f"error: {chamfer_over_time[current_time]:.2e} "
                            f"nof_pts: {len(pts_surface)}"
                            )

# Optional: Load the best model after training
# if best_model_state is not None:
#     model.load_state_dict(best_model_state)
#     print(f"Best model loaded with loss {best_loss}")

plt.plot(loss_over_time.keys(), loss_over_time.values())
plt.semilogy()
plt.show()

  0%|          | 0/100000 [00:00<?, ?it/s]

/opt/homebrew/Caskroom/miniforge/base/envs/ginns/lib/python3.12/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1741947704867/work/aten/src/ATen/native/TensorShape.cpp:3638.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


KeyboardInterrupt: 

### Visualize the result

In [6]:
# Generate the mesh using the level set function
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-0.85, -0.85, -0.85], dtype=torch.float64),
    bbox_max=torch.tensor([0.85, 0.85, 0.85], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)
fig.display()

Output()

In [7]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-0.85, -0.85, -0.85], dtype=torch.float64),
    bbox_max=torch.tensor([0.85, 0.85, 0.85], dtype=torch.float64),
    chunks=2
)

model.double()
from torch.func import vmap

# Vectorized mean curvature calculation
mean_curvatures = vmap(res_lib._mean_curvature, in_dims=(None, 0))(
    model.params, torch.tensor(verts, dtype=torch.float64)
).squeeze()

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)

color_map = k3d.basic_color_maps.Jet
color_range = [0, 0.005]

fig += k3d.mesh(
    verts, faces, 
    attribute=mean_curvatures.cpu().detach().numpy().astype(np.float32),
    color_range=color_range,
    color_map=color_map,
    side='double',
    flat_shading=False
)

fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()

Output()